# Simulated-participant dialogue-act analysis (2 groups)

The LLM-run counterpart of `othello_data_analysis_2group.ipynb` (humans). Same
pipeline — annotate every assisted-round user turn with the `DialogueActSuite`
panel, split runs into two outcome groups, compare act usage — over
`agentic_sim.py` output instead of `recordings-download/`.

**What maps onto what**

| human | simulated run |
|---|---|
| `recordings-download/<pid>/` | `test_agentic_run[_<game>]/preset_<n><model>/` |
| `oth_score_p*.json` → `["score"]` | `summary_p*.json` → `["optimal_moves"]` |
| `conversation_p<puzzle>.jsonl` | same filename, same `user`/`assistant` keys |
| `demographics.json` | `summary["profile"]` (one of 5 fixed presets) |
| `post_survey`, NASA-TLX, `assistant_helpfulness.csv` | no analogue — those cells are dropped |

**Deliberately not ported**

- *NASA-TLX / post-survey / assistant-helpfulness* — a simulated student fills in
  no surveys.
- *Time per puzzle* — sim timestamps are synthetic (a fixed +5s assistant latency
  and +15s between turns, see `agentic_sim.py`), so any duration measured from
  them is a constant, not a behaviour.

**One unit-of-analysis caveat, carried all the way through:** a human `pid` is one
person. A run directory is one *persona × model* cell — 5 presets × ~12 models.
So "demographics" here vary over only 5 fixed profiles and are crossed with model
identity. Group differences can therefore come from the persona prompt, the model,
or their interaction; the last two cells are where that gets pulled apart.

In [ ]:
# --- Setup: all imports in one place ---
import glob
import importlib.util
import json
import os
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# --- Which game / which run root -------------------------------------------
# One switch. Everything downstream (paths, puzzle ids, which annotator prompt
# gets loaded) derives from GAME, so the whole notebook re-runs on Connect Four
# by editing this one line.
#
# Puzzle ids and round structure come straight from constants.GAMES, so they
# cannot drift from what the simulator actually played:
#   othello       round 0 = oc20260727 (assistant)  round 1 = b220260706 + bg20260726 (solo)
#   connect_four  round 0 = 15         (assistant)  round 1 = w4p6                    (solo)
GAME = "othello"                     # "othello" or "connect_four"

RUN_ROOT = Path("test_agentic_run" if GAME == "othello" else f"test_agentic_run_{GAME}")

# The annotator prompt is game-specific (its examples name squares vs columns).
# The two variants live in files whose names differ by a space, so they are loaded
# by PATH rather than by import.
ANNOTATOR_FILE = {"othello": "dialogue_act_annotation.py",
                  "connect_four": "dialogue_act_annotation copy.py"}[GAME]

_spec = importlib.util.spec_from_file_location("da_mod", ANNOTATOR_FILE)
da_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(da_mod)

# Puzzle ids, read from the simulator's own config.
_cspec = importlib.util.spec_from_file_location("sim_constants", "constants.py")
sim_constants = importlib.util.module_from_spec(_cspec)
_cspec.loader.exec_module(sim_constants)

ROUNDS = sim_constants.GAMES[GAME]["rounds"]
AI_PUZZLE = ROUNDS[0]["puzzles"][0]                 # the assisted round
SOLO_PUZZLES = ROUNDS[-1]["puzzles"]                # the no-assistant round(s)
ALL_PUZZLES = [AI_PUZZLE, *SOLO_PUZZLES]

CONVO_F = f"conversation_p{AI_PUZZLE}.jsonl"
ANNOTATED_F = f"annotated_conversation_p{AI_PUZZLE}.jsonl"

print(f"game={GAME}  root={RUN_ROOT}  assisted={AI_PUZZLE}  solo={SOLO_PUZZLES}")
print(f"annotator prompt: {ANNOTATOR_FILE}")
print(f"scheme: {list(da_mod.DialogueAct.__args__)}")

# Which runs count

The human notebook's `all_candidates` = participants with a score file for
*every* round. Same rule here: a run counts only if it wrote a `summary_p*.json`
for all of `ALL_PUZZLES`, i.e. it got through the whole study without dying on a
provider error.

A run with an **empty conversation** is kept in `all_runs` (it still has an
outcome) but has no turns to annotate — a simulated student that never called
`consult_assistant`. That is a behaviour, not a failure, so it is counted
separately rather than dropped silently.

In [ ]:
# --- Discover complete runs ------------------------------------------------
RUN_RE = re.compile(r"^preset_(\d+)(.+)$")


def parse_run(run_id):
    """'preset_2together_openai_gpt-oss-120b' -> (2, 'together/openai/gpt-oss-120b').

    agentic_sim.py builds the directory name by replacing ':' and '/' with '_',
    which is lossy — 'gpt-oss-120b' legitimately contains no separator to restore.
    So the model is returned with underscores intact and used only as a LABEL;
    nothing keys off a reconstructed provider path.
    """
    m = RUN_RE.match(run_id)
    if not m:
        return None, run_id
    return int(m.group(1)), m.group(2)


def summaries(run_id):
    """{puzzle_id: summary dict} for whichever summaries this run wrote."""
    out = {}
    for p in ALL_PUZZLES:
        f = RUN_ROOT / run_id / f"summary_p{p}.json"
        if f.exists():
            out[p] = json.load(open(f))
    return out


def convo_turns(run_id, name=None):
    """User+assistant exchanges in a run's conversation log (one per line)."""
    f = RUN_ROOT / run_id / (name or CONVO_F)
    if not f.exists():
        return []
    return [json.loads(l) for l in open(f) if l.strip()]


all_dirs = sorted(d.name for d in RUN_ROOT.iterdir() if d.is_dir())
all_runs = [r for r in all_dirs if len(summaries(r)) == len(ALL_PUZZLES)]
incomplete = [r for r in all_dirs if r not in set(all_runs)]

n_silent = sum(not convo_turns(r) for r in all_runs)
print(f"{len(all_dirs)} run dirs | {len(all_runs)} complete | {len(incomplete)} incomplete")
print(f"of the complete runs, {n_silent} asked the assistant nothing "
      f"({len(all_runs) - n_silent} have turns to annotate)")
if incomplete:
    print("\nincomplete (missing summaries — died mid-sweep, or still running):")
    for r in incomplete:
        print(f"  {r:58s} has {sorted(summaries(r))}")

# Outcome: continuous for Othello, two groups for Connect Four

Both games measure the **solo** rounds — performance *without* the assistant, the
thing the assisted round was supposed to teach. How that becomes an outcome
variable differs by game, and it is a real fork in the analysis, not a formatting
choice:

| game | `OUTCOME_MODE` | outcome | analysis |
|---|---|---|---|
| **othello** | `"margin"` | **sum of `final_margin` over both solo puzzles** — a signed disc difference, continuous | correlations of act share against margin |
| **connect_four** | `"groups"` | `optimal_moves >= 2` on the one solo puzzle, scored over the first 3 decisions | Solvers vs Strugglers, permutation tests |

**Why Othello is continuous.** Dichotomising a graded score throws away most of its
information and makes the result hinge on where the cut falls — which we saw
directly: three different Othello cut rules gave 84, 71 and 71 Solvers out of 222
and disagreed on 13 runs. The margin sum keeps the full ordering, so a run that lost
by 2 is distinguished from one that lost by 22. It also has more power at the same n,
which matters because the act shares are noisy.

The margin ceiling is the two puzzles' `optimal_margin` summed (12 + 4 = **16**), so
a run at +16 played both perfectly. Observed range on the current data is −36 to +16,
mean −6.0.

**Cost of the change, stated plainly:** correlations against a continuous margin have
no counterpart in the human Othello notebook, which splits into two groups. Othello
results here are therefore no longer a like-for-like comparison with the human
analysis — they answer a better-posed question about the same data. Connect Four
still mirrors its human split exactly, so that comparison is intact.

The group-based cells below skip themselves when `OUTCOME_MODE == "margin"`, and the
correlation cells skip when it is `"groups"`, so the notebook runs end to end on
either game without editing anything but `GAME`.

In [ ]:
# --- 2-group split on solo-round performance -------------------------------
GOOD_LABEL, BAD_LABEL = "won", "lost"        # same strings the human notebook uses
DISP = {"won": "Solvers", "lost": "Strugglers"}


# The human round is a FIXED THREE scored moves per puzzle (all 130 CF participants
# have num_moves == 3), but the simulator plays the puzzle out — Connect Four w4p6
# runs make anywhere from 2 to 10 decisions. Left alone, `summary["optimal_moves"]`
# counts optimal decisions out of a variable denominator, so "score >= 2" means
# 2-of-2 for one run and 4-of-10 for another, and the DV runs 0-7 where the human one
# runs 0-3. Truncating at the third decision puts both populations on the same scale.
# Othello is unaffected (its solo puzzles already stop at 2-3 decisions); this is a
# no-op there rather than a special case.
TRUNC_DECISIONS = 3


def puzzle_score(run_id, puzzle):
    """(optimal, decisions) over this puzzle's first TRUNC_DECISIONS decisions.

    Ordered by `decision_number`, not file order, so a re-attempt written out of
    sequence cannot change which moves count as the first three. Falls back to the
    summary if the move log is missing — that only over-counts when a run exceeded
    the truncation, so it fails loudly in the diagnostics rather than silently.
    """
    f = RUN_ROOT / run_id / f"moves_p{puzzle}.jsonl"
    if not f.exists():
        s = summaries(run_id)[puzzle]
        return s["optimal_moves"], s["decisions"]
    rows = sorted((json.loads(l) for l in open(f) if l.strip()),
                  key=lambda r: r.get("decision_number") or 0)[:TRUNC_DECISIONS]
    return sum(bool(r.get("optimal")) for r in rows), len(rows)


def solo_score(run_id):
    """(optimal decisions, scored decisions) summed over the solo puzzles."""
    per = [puzzle_score(run_id, p) for p in SOLO_PUZZLES]
    return sum(o for o, _ in per), sum(d for _, d in per)


def solo_wins(run_id):
    """How many of the solo puzzles this run actually won."""
    s = summaries(run_id)
    return sum(bool(s[p]["won"]) for p in SOLO_PUZZLES)


def solo_margin(run_id):
    """Signed disc difference summed over the solo puzzles — the continuous outcome.

    Othello only: `final_margin` is a disc count, and Connect Four is win/lose with no
    margin at all, so its summaries carry no such field. Returns NaN there rather than
    raising, which is what lets the same cell run on both games.
    """
    s = summaries(run_id)
    vals = [s[p].get("final_margin") for p in SOLO_PUZZLES]
    return float("nan") if any(v is None for v in vals) else sum(vals)


# Continuous margin for Othello, two groups for Connect Four. See the note above.
OUTCOME_MODE = {"othello": "margin", "connect_four": "groups"}[GAME]

# Only consulted when OUTCOME_MODE == "groups".
#   "abs_optimal" -- absolute cut of 2 optimal decisions per solo puzzle, the human
#                    rule (CF `flat_score >= 2`); the app `score` it uses is the count
#                    of optimal decisions, == optimal_moves over the first 3.
#   "won_any"     -- won at least one solo puzzle.
RULE = {"othello": "won_any", "connect_four": "abs_optimal"}[GAME]

ABS_CUT = 2 * len(SOLO_PUZZLES)
RATE_CUT = 2 / 3

rows = []
for r in all_runs:
    opt, dec = solo_score(r)
    rows.append({"run": r, "preset": parse_run(r)[0], "model": parse_run(r)[1],
                 "opt": opt, "dec": dec, "rate": opt / dec if dec else 0.0,
                 "wins": solo_wins(r), "margin": solo_margin(r),
                 "turns": len(convo_turns(r))})
scores = pd.DataFrame(rows).set_index("run")

# Groups are always COMPUTED (they are cheap, and the diagnostic below is the whole
# argument for not using them on Othello); OUTCOME_MODE decides what is USED.
scores["grp_won_any"] = np.where(scores["wins"] >= 1, GOOD_LABEL, BAD_LABEL)
scores["grp_abs"] = np.where(scores["opt"] >= ABS_CUT, GOOD_LABEL, BAD_LABEL)
scores["grp_rate"] = np.where(scores["rate"] >= RATE_CUT, GOOD_LABEL, BAD_LABEL)

RULE_COL = {"won_any": "grp_won_any", "abs_optimal": "grp_abs", "rate": "grp_rate"}[RULE]
RULE_DESC = {"won_any": f"won >= 1 of {len(SOLO_PUZZLES)} solo puzzle(s)",
             "abs_optimal": f"optimal decisions >= {ABS_CUT}",
             "rate": f">= {RATE_CUT:.2f} of decisions optimal"}[RULE]

print(f"won-any rule  (wins >= 1):        {dict(scores['grp_won_any'].value_counts())}")
print(f"absolute rule (opt >= {ABS_CUT}):        {dict(scores['grp_abs'].value_counts())}")
print(f"rate rule     (>= {RATE_CUT:.2f} optimal): {dict(scores['grp_rate'].value_counts())}")
for a, b in [("grp_won_any", "grp_abs"), ("grp_abs", "grp_rate")]:
    d = scores.index[scores[a] != scores[b]].tolist()
    print(f"  {a} vs {b}: differ on {len(d)} run(s)")

# Othello-only, like final_margin itself; None for a game with no margin.
_om = [summaries(all_runs[0])[p].get("optimal_margin") for p in SOLO_PUZZLES]
MARGIN_CEILING = None if any(v is None for v in _om) else sum(_om)

if OUTCOME_MODE == "margin" and scores["margin"].isna().all():
    raise RuntimeError(
        f"OUTCOME_MODE='margin' but no summary for {GAME} carries final_margin — "
        f"that field is Othello-specific. Use OUTCOME_MODE='groups' for this game.")

if OUTCOME_MODE == "groups":
    scores["outcome"] = scores[RULE_COL]
    good_runs = scores.index[scores["outcome"] == GOOD_LABEL].tolist()
    bad_runs = scores.index[scores["outcome"] == BAD_LABEL].tolist()
    group_of = scores["outcome"].to_dict()
    print(f"\nrule in force for {GAME}: {RULE!r} ({RULE_DESC})")
    print(f"Solvers n={len(good_runs)}   Strugglers n={len(bad_runs)}")
else:
    # Continuous outcome. good_runs/bad_runs stay EMPTY on purpose: the group cells
    # below test OUTCOME_MODE and skip, so an empty list is never silently plotted
    # as a group of size zero.
    good_runs, bad_runs, group_of = [], [], {}
    m = scores["margin"]
    print(f"\noutcome in force for {GAME}: continuous margin sum over {SOLO_PUZZLES}")
    print(f"  n={len(m)}  mean={m.mean():+.1f}  sd={m.std():.1f}  "
          f"median={m.median():+.1f}  range=[{m.min():+d},{m.max():+d}]")
    print(f"  ceiling (both puzzles optimal) = {MARGIN_CEILING:+d}; "
          f"{(m >= MARGIN_CEILING).sum()} run(s) reached it")
    print(f"  the three group rules above are reported only as a reference point — "
          f"none of them is used downstream")

scores.sort_values("margin", ascending=False)[
    ["preset", "model", "opt", "dec", "wins", "margin", "turns"]].head(10)

### Sanity check: is the split measuring the model, the persona, or the task?

Before reading anything into dialogue acts, look at where the outcome variance
actually sits. If `outcome` is essentially a function of `model`, then every
"Solvers vs Strugglers" difference below is a between-model comparison wearing a
learning-outcome label, and should be described that way.

In [ ]:
# --- Where does the outcome variance live: model or persona? ---------------
# OUTCOME_VAR is whatever this game measures, so the decomposition reads the same
# either way: solver rate for the grouped game, mean margin for the continuous one.
if OUTCOME_MODE == "groups":
    OUTCOME_VAR, OUTCOME_LAB = (scores[RULE_COL] == GOOD_LABEL).astype(float), "solver rate"
else:
    OUTCOME_VAR, OUTCOME_LAB = scores["margin"].astype(float), "mean margin"

tmp = scores.assign(_y=OUTCOME_VAR)
agg = dict(n=("_y", "size"), y=("_y", "mean"), mean_opt=("opt", "mean"),
           mean_turns=("turns", "mean"))
by_model = tmp.groupby("model").agg(**agg).sort_values("y", ascending=False)
by_preset = tmp.groupby("preset").agg(**agg)

print(f"=== by student model ===   (y = {OUTCOME_LAB})")
print(by_model.to_string(float_format=lambda v: f"{v:.2f}"))
print(f"\n=== by persona preset ===   (y = {OUTCOME_LAB})")
print(by_preset.to_string(float_format=lambda v: f"{v:.2f}"))
print(f"\nspread in {OUTCOME_LAB}:  across models {by_model['y'].std():.2f}"
      f"   across presets {by_preset['y'].std():.2f}")
# Variance explained, so the two factors are compared on one scale rather than by
# eyeballing two standard deviations computed over different numbers of groups.
tot = OUTCOME_VAR.var(ddof=0)
for fac in ("model", "preset"):
    grp = OUTCOME_VAR.groupby(scores[fac])
    between = ((grp.mean() - OUTCOME_VAR.mean()) ** 2 * grp.size()).sum() / len(OUTCOME_VAR)
    print(f"  eta^2 for {fac:<7}= {between / tot:.3f}" if tot else "")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
by_model["y"].plot.barh(ax=ax[0], color="#81dab6", edgecolor="white")
ax[0].set_title(f"{OUTCOME_LAB.title()} by student model"); ax[0].set_xlabel(OUTCOME_LAB)
ax[0].invert_yaxis(); ax[0].tick_params(labelsize=8)
by_preset["y"].plot.bar(ax=ax[1], color="#ef8649", edgecolor="white", rot=0)
ax[1].set_title(f"{OUTCOME_LAB.title()} by persona preset"); ax[1].set_xlabel("preset")
for a in ax:
    a.set_ylabel("")
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

# Conversation Differences

### Number of Turns

In [ ]:
# Port of the human `count_convo_turns`: mean/median/std exchanges in the
# assisted round. Runs that asked nothing count as 0, exactly as a participant
# with no conversation file did.
def count_convo_turns(runs):
    t = [len(convo_turns(r)) for r in runs]
    return np.mean(t), np.median(t), np.std(t)


print(f"all runs: mean/median/std turns = "
      f"{tuple(round(v, 2) for v in count_convo_turns(list(scores.index)))}")
print(f"silent runs (0 turns): {sum(not convo_turns(r) for r in scores.index)}"
      f"/{len(scores)}")

if OUTCOME_MODE == "groups":
    print(f"\nSOLVERS   ({RULE_DESC}): mean/median/std turns = "
          f"{tuple(round(v, 2) for v in count_convo_turns(good_runs))}")
    print(f"STRUGGLERS: mean/median/std turns = "
          f"{tuple(round(v, 2) for v in count_convo_turns(bad_runs))}")
    print(f"silent runs: Solvers "
          f"{sum(not convo_turns(r) for r in good_runs)}/{len(good_runs)}   "
          f"Strugglers {sum(not convo_turns(r) for r in bad_runs)}/{len(bad_runs)}")
else:
    # Spearman, not Pearson: turn counts are small integers with a long right tail
    # (a handful of runs ask 15+ questions), which a Pearson r would let dominate.
    rho, p = stats.spearmanr(scores["turns"], scores["margin"])
    print(f"\nturns vs margin: Spearman rho={rho:+.3f} p={p:.4f} (n={len(scores)})")
    print("  Sign check before reading the act correlations: if asking MORE associates "
          "with a\n  worse margin, that is mostly the weaker students needing to ask, "
          "not asking causing harm.")

### Dialogue Acts

Annotation is the expensive step: a 3-model panel per user turn, so it writes one
`annotated_conversation_p<puzzle>.jsonl` per run and **skips runs already
annotated**. Set `OVERWRITE = True` to force a re-run (e.g. after changing the
taxonomy or the prompt).

In [ ]:
# --- Annotate every assisted-round user turn with the panel ---------------
# Slow + hits APIs (3 models x every user turn), so it SKIPS runs already
# annotated. Set OVERWRITE = True to force a re-run.
import dotenv
dotenv.load_dotenv()

da = da_mod.DialogueActSuite()
print(da.annotators)
OVERWRITE = False

for n, run_id in enumerate(all_runs, 1):
    turns = convo_turns(run_id)
    if not turns:                                # never consulted the assistant
        continue
    out_f = RUN_ROOT / run_id / ANNOTATED_F
    if out_f.exists() and out_f.stat().st_size > 0 and not OVERWRITE:
        print(f"[{n}/{len(all_runs)}] {run_id} already annotated — skipping")
        continue

    print(f"[{n}/{len(all_runs)}] {run_id} annotating {len(turns)} turns...")
    with open(out_f, "w") as w:
        for curr_l in turns:
            utt = (curr_l.get("user") or "").strip()
            curr_l["annotation_user"] = da(utterance=utt) if utt else None
            w.write(json.dumps(curr_l) + "\n")

#### Did the panel agree?

Before reading act frequencies, check that the labels are stable — an act
distribution built from three models that disagree is measuring the annotators.
`dialogue_act_annotation.py` ships the two metrics for this, so they are reported
here rather than computed ad hoc:

- **mean pairwise Jaccard** — set overlap between two annotators' label sets,
  averaged over items and pairs. Intuitive, but inflated by the acts everyone
  finds easy.
- **Fleiss' κ (macro)** — each act treated as an independent present/absent
  rating, κ per act, averaged. Chance-corrected, so it is the one to quote.

Also worth watching: `n_valid` should be 3 on every turn. If a model is timing out
it silently shrinks the panel (fail-open by design), and a 2-annotator "majority"
is just one model agreeing with another.

In [ ]:
# --- Panel agreement over the annotated turns ------------------------------
records = []
for run_id in all_runs:
    f = RUN_ROOT / run_id / ANNOTATED_F
    if not f.exists():
        continue
    for l in open(f):
        ann = json.loads(l).get("annotation_user")
        if ann:
            records.append(ann["per_model"])

n_valid = Counter(len([v for v in r.values() if v is not None]) for r in records)
failures = Counter(k for r in records for k, v in r.items() if v is None)
print(f"{len(records)} annotated turns | n_valid distribution: {dict(n_valid)}")
print(f"annotator failures (dropped from the vote): {dict(failures) or 'none'}")

macro, by_act = da_mod.fleiss_kappa(records)
print(f"\nmean pairwise Jaccard: {da_mod.mean_pairwise_jaccard(records):.3f}")
print(f"Fleiss' kappa (macro): {macro:.3f}")
print("kappa by act:", {k: round(v, 2) for k, v in by_act.items()})

In [ ]:
# --- Tidy LONG dataframe: one row per (turn, dialogue act) ----------------
# Same shape as the human notebook's `df`, with `pid` renamed to `run`.
def normalize_ann(ann):
    """Canonicalize a stored annotation. Handles both the current dict form and
    the legacy tuple form [consensus, votes, per_model] (serialized as a list) so
    old annotated files don't need re-running."""
    if ann is None:
        return None
    if isinstance(ann, dict):
        return ann
    consensus, votes, per_model = ann                       # legacy list form
    n_valid = sum(v is not None for v in per_model.values())
    confidence = {a: votes[a] / n_valid for a in votes} if n_valid else {}
    no_majority = not consensus and bool(votes)
    final = consensus if consensus else [a for a in votes]  # no-majority -> union
    return {"final": final, "consensus": consensus, "votes": votes,
            "confidence": confidence, "per_model": per_model, "n_valid": n_valid,
            "no_majority": no_majority, "needs_review": no_majority or n_valid < 2,
            "arbiter_used": False}


# Manual relabels for acts that don't fit this student<->AI setting. Kept
# identical to the human notebook so the two act distributions are comparable —
# any change here has to be made in both or the comparison is meaningless.
ACT_REMAP = {"Correct Answer": "Think Aloud",
             "Social Coordination Action": "Metacomment",
             "Forced Choice": "Metacomment"}

rows = []
for run_id in all_runs:
    convo_f = RUN_ROOT / run_id / ANNOTATED_F
    if not convo_f.exists():
        continue
    preset, model = parse_run(run_id)
    for i, l in enumerate(open(convo_f)):
        curr_l = json.loads(l)
        ann = normalize_ann(curr_l.get("annotation_user"))
        acts = [ACT_REMAP.get(a, a) for a in (ann["final"] if ann else [])]
        acts = list(dict.fromkeys(acts))                      # dedup after remap
        conf = {ACT_REMAP.get(k, k): v for k, v in (ann.get("confidence", {}) if ann else {}).items()}
        needs_review = bool(ann and ann.get("needs_review"))
        # one row per act; keep turns with no acts as a single utt_type=None row
        for act in (acts or [None]):
            rows.append({
                "run": run_id,
                "preset": preset,
                "model": model,
                "utt": curr_l.get("user"),
                "utt_type": act,
                "conf": conf.get(act) if act else None,
                "needs_review": needs_review,
                "utt_ind": i,
                "outcome": group_of.get(run_id, "MISSING"),
            })

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError(
        f"no {ANNOTATED_F} files under {RUN_ROOT} — run the annotation cell above first "
        f"(or, if it was interrupted, re-run it: it resumes by skipping finished runs)")
print(f"{len(df)} rows | {df['run'].nunique()} runs | "
      f"{df['needs_review'].mean():.0%} of rows flagged needs_review")
df.head(n=5)

In [ ]:
df.loc[df["outcome"] == "won"]["utt_type"].value_counts()

In [ ]:
df.loc[df["outcome"] == "lost"]["utt_type"].value_counts()

### Dialogue-act frequency by outcome group

Rate = **acts of a given type per user turn**, computed per run then averaged
within each group so every run counts equally regardless of how many turns it
took. Error bars = SEM across runs.

Note the denominator: `turns_per_run` counts only runs that *have* annotated
turns. Silent runs (0 turns) contribute no acts and are absent from this chart —
they are a finding about whether the student consults at all, which the turn-count
cell above reports, not about which acts it uses.

In [ ]:
if OUTCOME_MODE != "groups":
    print("skipped — act-rate bars compare two groups; {GAME} uses the continuous "
          "margin outcome. See the correlation cells below.".format(GAME=GAME))
else:
    # Per-run, per-turn rate of each dialogue act, averaged within group.
    turns_per_run = df.groupby("run")["utt_ind"].nunique()          # user turns per run
    acts_df = df.dropna(subset=["utt_type"])

    cnt = acts_df.groupby(["run", "utt_type"]).size().rename("n").reset_index()
    cnt["rate"] = cnt["n"] / cnt["run"].map(turns_per_run)

    # runs x acts table of rates (0 where a run never used that act); reindex over
    # ALL runs in df so runs with no coded acts still count as zeros
    rate_tbl = cnt.pivot_table(index="run", columns="utt_type", values="rate", fill_value=0)
    rate_tbl = rate_tbl.reindex(sorted(df["run"].unique()), fill_value=0)
    rate_tbl["outcome"] = rate_tbl.index.map(group_of)

    acts_order = rate_tbl.drop(columns="outcome").sum().sort_values(ascending=False).index.tolist()
    grp_mean = rate_tbl.groupby("outcome")[acts_order].mean()
    grp_sem = rate_tbl.groupby("outcome")[acts_order].sem()
    n_by = rate_tbl["outcome"].value_counts()

    GRP = [("won", "#81dab6"), ("lost", "#ef8649")]
    fig, ax = plt.subplots(figsize=(max(9, 0.9 * len(acts_order) + 3), 5))
    x = np.arange(len(acts_order)); w = 0.26
    for gi, (g, col) in enumerate(GRP):
        if g not in grp_mean.index:
            continue
        ax.bar(x + (gi - 0.5) * w, grp_mean.loc[g], w, yerr=grp_sem.loc[g], capsize=3,
               color=col, label=f"{DISP.get(g, g)} (n={n_by.get(g, 0)})", edgecolor="white", lw=0.5)
    ax.set_xticks(x); ax.set_xticklabels(acts_order, rotation=40, ha="right", fontsize=8)
    ax.set_ylabel("mean acts per user turn")
    ax.set_title(f"Dialogue-act frequency by outcome group, simulated students ({GAME})")
    ax.legend(); plt.tight_layout(); plt.show()

### How far apart the two outcomes sit (dumbbell)

One row per dialogue act: a dot for **Solvers** and one for **Strugglers**, each =
that group's **mean share of a run's acts (%)**, joined by a gray bar whose length
is the gap. **Longer bar = the act separates the groups more.** Right-hand column
is a two-sided Mann-Whitney U on the per-run shares.

With ~30 runs and a handful of acts this is exploratory in exactly the way the
human version is, and the Bonferroni line in the footnote is the honest threshold.

In [ ]:
if OUTCOME_MODE != "groups":
    print("skipped — dumbbell compare two groups; {GAME} uses the continuous "
          "margin outcome. See the correlation cells below.".format(GAME=GAME))
else:
    # Dumbbell: Solvers vs Strugglers separation on each dialogue act (+ significance).
    # metric = mean share of a run's acts (%). Labels sit on the OUTER side of each dot
    # (smaller->left, larger->right) so they never overlap when dots are close.
    from scipy.stats import mannwhitneyu
    import matplotlib.transforms as mtransforms

    LAB = [("Solvers", good_runs, "#22c55e"), ("Strugglers", bad_runs, "#ef4444")]
    acts_all = df.dropna(subset=["utt_type"])
    tot = acts_all.groupby("run").size()
    cnt = (acts_all.groupby(["run", "utt_type"]).size().unstack(fill_value=0)
             .reindex(sorted(df["run"].unique()), fill_value=0))
    share = 100 * cnt.div(tot, axis=0).fillna(0)                 # run x act (% of that run's acts)

    won_ids = [r for r in good_runs if r in share.index]
    lost_ids = [r for r in bad_runs if r in share.index]


    def gmean(runs, a):
        idx = [r for r in runs if r in share.index]
        return share.loc[idx, a].mean() if idx else np.nan


    def mwu_p(a):
        _, p = mannwhitneyu(share.loc[won_ids, a].values,
                            share.loc[lost_ids, a].values, alternative="two-sided")
        return p


    def sig_mark(p):
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        if p < 0.10:  return "†"                            # dagger = trend
        return "n.s."


    acts = sorted(share.columns, key=lambda a: abs(gmean(good_runs, a) - gmean(bad_runs, a)), reverse=True)
    y = np.arange(len(acts))
    PAD = 1.4                                                     # horizontal gap between dot and its label
    alpha_bonf = 0.05 / len(acts)                                # multiple-comparison threshold

    fig, ax = plt.subplots(figsize=(10.5, 0.7 * len(acts) + 2))
    XR = max(66, share.mean().max() + 22)                        # room for the significance column
    SIG_X = XR - 13                                              # left edge of the p-value column
    for yi, a in zip(y, acts):
        ranked = sorted(LAB, key=lambda t: gmean(t[1], a))       # ascending by value
        (lo_lab, lo_p, lo_c), (hi_lab, hi_p, hi_c) = ranked
        lo, hi = gmean(lo_p, a), gmean(hi_p, a)
        ax.hlines(yi, lo, hi, color="#d9d9d9", lw=6, zorder=1, capstyle="round")
        for lab, runs_, col in LAB:
            ax.scatter(gmean(runs_, a), yi, s=230, color=col, edgecolor="white", lw=1.5, zorder=3)
        ax.text(lo - PAD, yi, f"{lo:.0f}", ha="right", va="center", fontsize=9, color=lo_c)   # smaller -> left
        ax.text(hi + PAD, yi, f"{hi:.0f}", ha="left", va="center", fontsize=9, color=hi_c)    # larger  -> right
        p = mwu_p(a)
        is_sig = p < 0.05
        ax.text(SIG_X, yi, f"{sig_mark(p):>4}  p={p:.2f}", ha="left", va="center", fontsize=9,
                family="monospace", color=("#111" if is_sig else "#999"),
                fontweight=("bold" if is_sig else "normal"))
    ax.set_yticks(y); ax.set_yticklabels(acts, fontsize=11)
    ax.invert_yaxis()
    ax.set_xlim(-6, XR)
    ax.set_xticks([t for t in range(0, 51, 10)])                 # ticks span the data range only
    ax.set_xlabel("Mean share of a run's acts (%)")
    ax.set_title(f"Simulated students: Solvers vs. Strugglers ({GAME})",
                 fontsize=15, fontweight="bold", loc="left", pad=26)
    trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.text(SIG_X, 1.005, "Solvers vs Strugglers\n(Mann-Whitney U)", transform=trans, ha="left", va="bottom",
            fontsize=8.5, color="#555", fontweight="bold")
    ax.grid(axis="x", color="#eee", zorder=0); ax.set_axisbelow(True)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.tick_params(length=0)
    handles = [plt.Line2D([0], [0], marker="o", ls="", ms=11, mfc=col, mec="white",
                          label=f"{lab} (n={len([r for r in runs_ if r in share.index])})")
               for lab, runs_, col in LAB]
    ax.legend(handles=handles, loc="lower left", frameon=False, fontsize=10, ncol=2,
              bbox_to_anchor=(0.28, 0.02))
    fig.text(0.01, -0.03,
             f"Markers: *** p<.001   ** p<.01   * p<.05   † p<.10 (trend)   n.s. = not significant   "
             f"(two-sided Mann-Whitney U on each act's per-run share).\n"
             f"n = {len(won_ids)} Solvers vs {len(lost_ids)} Strugglers; {len(acts)} acts tested, "
             f"Bonferroni α = {alpha_bonf:.3f}. Exploratory. NOTE: runs are persona x model cells, "
             f"not independent people — see the variance-decomposition cell.",
             fontsize=8, color="#777", ha="left")
    plt.tight_layout(); plt.show()

### Solvers vs Strugglers: permutation test per dialogue act

Label-permutation test on each act's **share of a run's acts (%)**, plus
Mann-Whitney U and Cohen's d. Same reasoning as the human notebook for preferring
a permutation test to Welch's t: these shares are bounded, zero-heavy proportions
built from very small per-run act counts, so normality is not a safe assumption.

In [ ]:
if OUTCOME_MODE != "groups":
    print("skipped — permutation tests compare two groups; {GAME} uses the continuous "
          "margin outcome. See the correlation cells below.".format(GAME=GAME))
else:
    # Permutation test (+ Mann-Whitney, Cohen's d) on per-run act share.
    from scipy import stats

    acts_all = df.dropna(subset=["utt_type"])
    tot = acts_all.groupby("run").size()
    cnt = (acts_all.groupby(["run", "utt_type"]).size().unstack(fill_value=0)
             .reindex(sorted(df["run"].unique()), fill_value=0))
    share = 100 * cnt.div(tot, axis=0).fillna(0)                 # run x act (% of that run's acts)
    won_ids = [r for r in good_runs if r in share.index]
    lost_ids = [r for r in bad_runs if r in share.index]

    # Combined "reasoning" acts proportion (share of a run's acts that are any of these).
    REASONING_ACTS = ["Think Aloud"]
    reasoning_share = share[[a for a in REASONING_ACTS if a in share.columns]].sum(axis=1)

    variables = [(a, share[a]) for a in share.columns]
    variables.append(("Reasoning (TA)", reasoning_share))


    def cohend(x, z):
        nx, nz = len(x), len(z)
        sp = np.sqrt(((nx - 1) * x.var(ddof=1) + (nz - 1) * z.var(ddof=1)) / (nx + nz - 2))
        return (x.mean() - z.mean()) / sp if sp > 0 else np.nan


    def perm_test(x, z, n_perm=20000, seed=0):
        """Two-sided permutation test on the difference in group means (label shuffling)."""
        obs = x.mean() - z.mean()
        pooled = np.concatenate([x, z]); nx = len(x)
        rng = np.random.default_rng(seed)
        ge = 0
        for _ in range(n_perm):
            perm = rng.permutation(pooled)
            d = perm[:nx].mean() - perm[nx:].mean()
            if abs(d) >= abs(obs) - 1e-12:
                ge += 1
        return obs, (ge + 1) / (n_perm + 1)                      # +1 = the observed labeling


    rows = []
    for a, series in variables:
        x, z = series.loc[won_ids].values, series.loc[lost_ids].values
        obs, p_perm = perm_test(x, z)                            # permutation (mean diff)
        _, pu = stats.mannwhitneyu(x, z, alternative="two-sided")
        rows.append((a, x.mean(), z.mean(), obs, p_perm, pu, cohend(x, z)))
    rows.sort(key=lambda r: r[4])                                # by permutation p

    alpha_bonf = 0.05 / len(rows)
    print(f"Permutation test (mean diff, 20k shuffles): Solvers (n={len(won_ids)}) vs "
          f"Strugglers (n={len(lost_ids)})   Bonferroni alpha = {alpha_bonf:.4f}\n")
    print(f"{'act':>28} {'Solv%':>6} {'Strug%':>6} {'diff':>6} {'perm_p':>7} {'MWU_p':>7} {'d':>6}")
    for a, w, l, dff, p, pu, d in rows:
        star = "*" if p < alpha_bonf else ("." if p < 0.05 else " ")
        print(f"{a:>28} {w:6.1f} {l:6.1f} {dff:+6.1f} {p:7.3f} {pu:7.3f} {d:+6.2f} {star}")
    print("\n  * p < Bonferroni alpha    . p < 0.05 (uncorrected)")

### Act share vs the continuous margin (Othello)

The counterpart of the group tests above, for the game that has a graded outcome.
For each act: **Spearman** rho between a run's act share (% of that run's acts) and
its solo-round margin sum, with a label-permutation p and BH-FDR q across acts.

Spearman rather than Pearson because act shares are bounded, zero-heavy proportions
and the margin has a hard ceiling at the optimal value — a handful of runs sitting
exactly at the ceiling would carry a Pearson r.

**`rho` is the headline estimate**, and it is the one comparable to the human
analysis. It is deliberately NOT adjusted for student model. Human participants
differ in ability, and the human notebook does not partial that out either; in this
simulator the student model is the instrument for that same variation, since a
single model prompted with different personas does not convincingly span cognitive
levels. Partialling out model would therefore remove the very variance that stands
in for human individual differences — it would over-control, not de-confound.

What model identity *also* carries is house style: verbosity, register, and how
many acts a model packs into a turn. A more able person does not automatically emit
2.4 acts per turn. So the diagnostic that matters is not "does this survive within
model" but **"is this carried by one engine"**:

- **rho_min (LOO)** — the weakest correlation obtained by dropping any single
  student model and recomputing, with the model whose removal causes it. An effect
  that holds across the suite stays significant here. One that collapses is that
  model's formatting quirk wearing an outcome label — as Knowledge Deficit Question
  does, going from −0.34 to −0.01 once Llama (0% Think Aloud, 36% KDQ, exactly 1.00
  acts/turn, worst margin) is removed.

In [ ]:
if OUTCOME_MODE != "margin":
    print(f"skipped — these correlations need the continuous outcome; "
          f"{GAME} uses two groups. See the permutation tests above.")
else:
    acts_all = df.dropna(subset=["utt_type"])
    tot = acts_all.groupby("run").size()
    cnt = (acts_all.groupby(["run", "utt_type"]).size().unstack(fill_value=0)
             .reindex(sorted(df["run"].unique()), fill_value=0))
    share = 100 * cnt.div(tot, axis=0).fillna(0)              # run x act (% of that run's acts)

    y = scores["margin"].reindex(share.index).astype(float)
    mdl = scores["model"].reindex(share.index)
    n_t = scores["turns"].reindex(share.index).astype(float)

    def perm_rho(x, yy, n_perm=20000, seed=0):
        """Permutation p for Spearman rho by shuffling the outcome labels."""
        r_obs = stats.spearmanr(x, yy).statistic
        rng = np.random.default_rng(seed)
        yv = np.asarray(yy, float)
        ge = sum(abs(stats.spearmanr(x, rng.permutation(yv)).statistic) >= abs(r_obs) - 1e-12
                 for _ in range(n_perm))
        return r_obs, (ge + 1) / (n_perm + 1)

    def loo_min(x, yy, grp):
        """Weakest rho over leave-one-model-out refits, and which model caused it.

        Robustness under a design where models ARE the ability variation: it asks
        whether the association survives without any one engine, instead of removing
        between-model variance wholesale (which would delete the intended signal).
        """
        d = pd.DataFrame({"x": x, "y": yy, "g": grp}).dropna()
        worst = (np.inf, np.nan, None)
        for m in d["g"].unique():
            s = d[d.g != m]
            if len(s) < 10 or s["x"].std() == 0 or s["y"].std() == 0:
                continue
            r, p = stats.spearmanr(s["x"], s["y"])
            if abs(r) < abs(worst[0]):
                worst = (r, p, m)
        return worst if np.isfinite(worst[0]) else (np.nan, np.nan, None)

    rows = []
    for a in share.columns:
        if share[a].std() == 0:
            continue
        rho, p = perm_rho(share[a].values, y.values)
        rw, pw, worst_m = loo_min(share[a], y, mdl)
        rp, _ = stats.spearmanr(share[a], n_t)                # act share vs how much they talked
        rows.append([a, share[a].mean(), rho, p, rw, pw, rp, worst_m])
    q = ([np.nan] * len(rows) if not rows else
         (lambda pv: [min(1.0, v) for v in
                      pd.Series(pv).rank(method="first").pipe(
                          lambda rk: pd.Series(pv) * len(pv) / rk).cummax()])(
             [r[3] for r in sorted(rows, key=lambda r: r[3])]))
    rows.sort(key=lambda r: r[3])

    print(f"n={len(share)} runs | outcome = margin sum over {SOLO_PUZZLES} "
          f"(ceiling {MARGIN_CEILING:+d})\n")
    print(f"{'act':>28} {'share%':>7} {'rho':>7} {'perm_p':>8} {'q':>6} "
          f"{'rho_min(LOO)':>13} {'p':>7} {'rho~turns':>10}  dropping")
    for (a, sh, rho, p, rw, pw, rp, wm), qq in zip(rows, q):
        star = "*" if qq < 0.05 else ("." if p < 0.05 else " ")
        f = lambda v: f"{v:+.3f}" if np.isfinite(v) else "    n/a"
        # Only an act that WAS significant can collapse; flagging an already-null act
        # as collapsing reads as evidence of an artifact where there was no effect.
        collapsed = qq < 0.05 and not (pw < 0.05)
        frag = "" if wm is None else (wm[:26] + ("   <-- COLLAPSES" if collapsed else ""))
        print(f"{a:>28} {sh:7.1f} {rho:+7.3f} {p:8.3f} {qq:6.3f} "
              f"{f(rw):>13} {pw:7.3f} {rp:+10.3f} {star} {frag}")
    print("\n  * BH-FDR q < 0.05    . uncorrected p < 0.05")
    print("  rho          = headline, NOT adjusted for model (models are the ability variation)")
    print("  rho_min(LOO) = weakest rho after dropping any one model; 'COLLAPSES' means the")
    print("                 association is one engine's house style, not a behavioural finding")
    print("  rho~turns    = act share vs turn count, the other thing shares track mechanically")

In [ ]:
if OUTCOME_MODE != "margin":
    print(f"skipped — scatter is for the continuous outcome; {GAME} uses two groups.")
else:
    top = [r[0] for r in rows[:4]]
    fig, ax = plt.subplots(1, len(top), figsize=(4 * len(top), 3.6), sharey=True)
    ax = np.atleast_1d(ax)
    for a, axi in zip(top, ax):
        axi.scatter(share[a], y, s=18, alpha=0.5, color="#3b82f6", edgecolor="none")
        if share[a].std() > 0:                      # LOWESS-free trend: simple OLS line
            b, a0 = np.polyfit(share[a], y, 1)
            xs = np.linspace(share[a].min(), share[a].max(), 50)
            axi.plot(xs, a0 + b * xs, color="#ef4444", lw=1.5)
        axi.axhline(MARGIN_CEILING, color="#999", ls=":", lw=1)
        axi.set_title(f"{a}\nrho={stats.spearmanr(share[a], y).statistic:+.2f}", fontsize=9)
        axi.set_xlabel("share of run's acts (%)")
        for s in ("top", "right"):
            axi.spines[s].set_visible(False)
    ax[0].set_ylabel(f"solo margin sum")
    fig.suptitle(f"Act share vs margin — {GAME} (dotted line = optimal {MARGIN_CEILING:+d})",
                 fontsize=11)
    plt.tight_layout(); plt.show()

In [ ]:
df.to_csv(f"simulated_utterances_{GAME}.csv", index=False)
print(f"wrote simulated_utterances_{GAME}.csv  ({len(df)} rows)")

---

# Left for you — two cells that are analysis design, not a port

Everything above is the human notebook re-pointed at simulated runs. The two
questions below have no counterpart to copy from, and they are where the
interesting decisions are. Signatures and the reasoning are laid out; the bodies
are yours.

### A. Persona vs model: which one drives act usage?

`outcome` may be close to a relabelling of `model` (the sanity-check cell above
tells you how close). If so, "Solvers use more Think Aloud" could just mean
"Opus talks more than Llama". You have a **crossed design** — 5 presets × ~12
models, roughly one run per cell — so this is answerable rather than a caveat.

Things to decide, in the order they bite:

1. **Unit and denominator.** Per-run act *share* (what the plots above use) or
   *rate per turn*? Share is closed-form compositional — the components sum to
   100%, so one act rising forces others down. Rate per turn is not. Which one
   makes a "model effect" mean what you want it to mean?
2. **Model.** With ~1 observation per (preset, model) cell you cannot fit an
   interaction *and* both main effects. Options: variance components on
   `act_share ~ preset + model` via the between-group sums of squares; a mixed
   model with `model` and `preset` as random intercepts (`statsmodels`
   `MixedLM` handles one grouping factor cleanly, two is awkward); or the
   cheap, honest version — a permutation test that shuffles labels *within*
   model, which asks "does persona move acts, holding the model fixed?"
3. **What would falsify the persona story?** Write that down before running it.

```python
def variance_by_factor(share_tbl, meta, factors=("preset", "model")):
    """Share of each act's between-run variance attributable to each factor.

    share_tbl : runs x acts (%)   -- the `share` frame built above
    meta      : runs x [preset, model]
    returns   : acts x factors frame of variance shares (eta-squared-like)
    """
    ...


def perm_within(share_tbl, meta, act, factor="model", by="outcome", n_perm=20000, seed=0):
    """Permutation test on the `by` group difference in `act`, shuffling labels
    only WITHIN each level of `factor`. Isolates the outcome effect from the
    between-model effect. Returns (observed_diff, p)."""
    ...
```

### B. Do simulated students talk like the humans did?

The point of the simulator. The human side already exports
`participant_utterances.csv` from cell 32 of `othello_data_analysis_2group.ipynb`;
this notebook writes `simulated_utterances_<game>.csv` in the same shape.

Before comparing distributions, three things have to be true, and none of them is
free:

1. **The act inventory must match.** Same `SCHEME`, same `ACT_REMAP`, same
   annotator panel and prompt version. If the human files were annotated with an
   older taxonomy, the comparison is measuring the prompt, not the students.
   Check this rather than assume it — `Counter` over `utt_type` in both files.
2. **Turn counts differ a lot** (humans ~4–8 exchanges, simulated runs 0–17 with
   a 10-question budget). So compare *shares*, and report the turn-count
   distributions alongside so a reader can see the mismatch you normalised away.
3. **What is the null?** "Simulated ≈ human" is a claim of *similarity*, and a
   non-significant test does not establish it. Either pick an equivalence bound
   you would accept in advance, or report a distance (e.g. Jensen–Shannon over
   the act distributions) with a bootstrap CI and let the number speak.

```python
def load_human_acts(path="participant_utterances.csv"):
    """Human long-form act rows, columns aligned to this notebook's `df`
    (`pid` -> `run`). Assert the act inventories match before returning."""
    ...


def act_distribution_distance(human_df, sim_df, n_boot=2000, seed=0):
    """Jensen-Shannon distance between the two act distributions, with a
    bootstrap CI over participants/runs. Returns (jsd, lo, hi)."""
    ...
```